# Musubi Tuner — LoRA Training on Google Colab

Train LoRA models for **HunyuanVideo**, **Wan2.1 / Wan2.2**, and other architectures using [musubi-tuner](https://github.com/kohya-ss/musubi-tuner) by kohya-ss.

---

## Quick Start Checklist

1. **Runtime → Change runtime type → GPU** (T4 for image LoRA, A100 for video LoRA)
2. Run **Section 1** – GPU check
3. Run **Section 2** – Mount Google Drive
4. Run **Section 3** – Install musubi-tuner
5. Edit **Section 4** – Set your config (architecture, paths, training params)
6. Run **Section 5** – Create directories
7. Run **Section 6** – Download models
8. Upload your dataset images + captions to Google Drive, then run **Section 7**
9. Run **Section 8** – Cache latents
10. Run **Section 9** – Cache text encoder outputs
11. Run **Section 10** – Train!
12. (Optional) Run **Section 11** – Test inference

---

## GPU & VRAM Guide

| Architecture | Task | Min VRAM | Recommended |
|---|---|---|---|
| Wan2.1 T2V 1.3B | Image LoRA | 12 GB | T4 (16 GB) |
| Wan2.1 T2V 14B | Image LoRA | 16 GB | A100 (40 GB) |
| HunyuanVideo | Image LoRA | 12 GB (fp8+swap) | A100 (40 GB) |
| Any | Video LoRA | 24 GB | A100 (40 GB) |

> **Tip**: For Colab free tier (T4, 16 GB), use **Wan2.1 T2V 1.3B** with image datasets for best results.


## Section 1 — GPU Check


In [ ]:
import subprocess, sys

# Show GPU info
!nvidia-smi

import torch
print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1024**3
    print(f"GPU             : {props.name}")
    print(f"VRAM            : {vram_gb:.1f} GB")
    if vram_gb < 14:
        print("\n⚠️  Less than 14 GB VRAM detected. Training may OOM.")
        print("   Use Wan2.1 T2V 1.3B with image data and all memory-saving options.")
    elif vram_gb < 24:
        print("\n✅ T4 / 16 GB detected. Good for Wan2.1 1.3B image LoRA training.")
    else:
        print("\n✅ A100 / 40+ GB detected. All architectures supported.")
else:
    print("\n❌ No GPU detected! Change runtime to GPU before proceeding.")


## Section 2 — Mount Google Drive

Models and outputs are saved to Google Drive so they persist across Colab sessions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted at /content/drive")


## Section 3 — Install musubi-tuner

Clones the repo and installs all dependencies. Takes ~3-5 minutes on first run.
Safe to skip if the `/content/musubi-tuner` directory already exists.


In [ ]:
import os

REPO_DIR = "/content/musubi-tuner"

if not os.path.exists(REPO_DIR):
    print("Cloning musubi-tuner...")
    !git clone https://github.com/kohya-ss/musubi-tuner {REPO_DIR}
else:
    print("Repo already cloned. Pulling latest changes...")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

# Upgrade pip silently
!pip install -q --upgrade pip

# Install PyTorch (skip if already correct version)
import torch
torch_version = tuple(int(x) for x in torch.__version__.split('.')[:2])
if torch_version < (2, 5):
    print("Upgrading PyTorch to 2.5.1+...")
    !pip install -q torch==2.5.1 torchvision --index-url https://download.pytorch.org/whl/cu124
else:
    print(f"PyTorch {torch.__version__} is compatible.")

# Install musubi-tuner and dependencies
print("Installing musubi-tuner...")
!pip install -q -e .

# Optional but useful packages
print("Installing optional packages...")
!pip install -q bitsandbytes tensorboard matplotlib ascii-magic

print("\n✅ Installation complete!")


## Section 4 — Configuration

**Edit this cell** to configure your training. All subsequent cells read from these variables.

### Architecture options:
| Value | Description | VRAM |
|---|---|---|
| `"wan2.1_t2v_1.3B"` | Wan2.1 Text-to-Video 1.3B (recommended for T4) | 12-16 GB |
| `"wan2.1_t2v_14B"` | Wan2.1 Text-to-Video 14B | 24+ GB |
| `"wan2.1_i2v_14B_480p"` | Wan2.1 Image-to-Video 14B 480p | 24+ GB |
| `"wan2.1_i2v_14B_720p"` | Wan2.1 Image-to-Video 14B 720p | 40+ GB |
| `"hunyuan_video"` | HunyuanVideo (fp8 recommended on T4) | 16+ GB |


In [ ]:
# ============================================================
#   ARCHITECTURE
# ============================================================
ARCHITECTURE = "wan2.1_t2v_1.3B"   # see options above

# ============================================================
#   PATHS  (all relative to your Google Drive)
# ============================================================
DRIVE_ROOT  = "/content/drive/MyDrive/musubi-tuner"
MODEL_DIR   = f"{DRIVE_ROOT}/models"        # downloaded model weights
DATASET_DIR = f"{DRIVE_ROOT}/dataset"       # your training images + captions
CACHE_DIR   = f"{DRIVE_ROOT}/cache"         # pre-cached latents / text encodings
OUTPUT_DIR  = f"{DRIVE_ROOT}/outputs"       # saved LoRA checkpoints

LORA_NAME   = "my-lora"                     # output file name (no extension)

# ============================================================
#   DATASET
# ============================================================
# Put your training images (JPG/PNG) and matching .txt captions
# in DATASET_DIR. Example:
#   /content/drive/MyDrive/musubi-tuner/dataset/image001.jpg
#   /content/drive/MyDrive/musubi-tuner/dataset/image001.txt

DATASET_TYPE    = "image"       # "image" or "video"
CAPTION_EXT     = ".txt"        # caption file extension
RESOLUTION      = [960, 544]    # [W, H] — reduce to [512, 512] for T4 with 14B
BATCH_SIZE      = 1             # increase if you have VRAM headroom
NUM_REPEATS     = 1             # repeat dataset N times (useful with tiny datasets)
ENABLE_BUCKET   = True          # aspect-ratio bucketing (recommended)

# Video-only settings (ignored for image datasets)
TARGET_FRAMES   = [1, 25]       # [1] for image-only, add more for video clips
FRAME_EXTRACTION = "head"       # "head", "chunk", "full", "slide", "uniform"
MAX_FRAMES      = 25

# ============================================================
#   LoRA PARAMETERS
# ============================================================
NETWORK_DIM     = 32            # LoRA rank (8-64, higher = more capacity)
NETWORK_ALPHA   = 16            # LoRA alpha (usually NETWORK_DIM / 2)

# ============================================================
#   TRAINING PARAMETERS
# ============================================================
LEARNING_RATE       = 2e-4
MAX_TRAIN_EPOCHS    = 16
SAVE_EVERY_N_EPOCHS = 1
SEED                = 42
OPTIMIZER_TYPE      = "adamw8bit"   # adamw8bit saves VRAM; use "adamw" if issues
MIXED_PRECISION     = "bf16"        # bf16 recommended; use fp16 if bf16 unsupported
TIMESTEP_SAMPLING   = "shift"
DISCRETE_FLOW_SHIFT = 3.0           # 7.0 for HunyuanVideo, 3.0 for Wan2.1

# ============================================================
#   MEMORY OPTIMIZATION
# ============================================================
USE_FP8_BASE                     = True   # fp8 DiT (saves ~50% VRAM, slight quality loss)
GRADIENT_CHECKPOINTING           = True   # always recommended
GRADIENT_CHECKPOINTING_CPU_OFFLOAD = False  # set True only if still OOM after fp8+swap
BLOCKS_TO_SWAP                   = 0      # 0=disabled; max 36 (HV) / 39 (Wan 14B)
                                           # increase if OOM during training

# ============================================================
#   LOGGING (optional)
# ============================================================
USE_TENSORBOARD = False          # set True to enable TensorBoard logging

print("✅ Configuration loaded.")
print(f"   Architecture : {ARCHITECTURE}")
print(f"   Dataset      : {DATASET_DIR}")
print(f"   Output       : {OUTPUT_DIR}/{LORA_NAME}")
print(f"   LoRA dim     : {NETWORK_DIM}")
print(f"   Epochs       : {MAX_TRAIN_EPOCHS}")


## Section 5 — Create Directories


In [ ]:
import os

for d in [MODEL_DIR, DATASET_DIR, CACHE_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"  ✅ {d}")

print("\nDirectories ready.")


## Section 6 — Download Models

Downloads the required model weights to Google Drive.
**Skip any file you already have** by setting the corresponding `SKIP_*` flag to `True`.

### File sizes (approximate)

| File | Size |
|---|---|
| Wan2.1 VAE | ~0.5 GB |
| Wan2.1 T5 text encoder | ~11 GB |
| Wan2.1 DiT 1.3B (bf16) | ~3 GB |
| Wan2.1 DiT 14B (bf16) | ~28 GB |
| HunyuanVideo DiT (fp8) | ~13 GB |
| HunyuanVideo VAE | ~0.2 GB |
| HunyuanVideo LLM encoder | ~8.5 GB |
| HunyuanVideo CLIP encoder | ~0.5 GB |

> Models are saved to Google Drive and reused across sessions.


In [ ]:
from huggingface_hub import hf_hub_download
import os

# Set True to skip re-downloading a file that already exists
SKIP_IF_EXISTS = True

def download(repo_id, filename, local_dir, subfolder=None):
    dest = os.path.join(local_dir, filename)
    if SKIP_IF_EXISTS and os.path.exists(dest):
        print(f"  ⏭️  Skipping (exists): {filename}")
        return dest
    print(f"  ⬇️  Downloading: {filename} from {repo_id}")
    path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        subfolder=subfolder,
        local_dir=local_dir,
    )
    print(f"  ✅ Saved to: {path}")
    return path


# ---- Architecture-specific download ----
if ARCHITECTURE.startswith("wan2.1"):
    print("=== Downloading Wan2.1 models ===")

    # VAE (shared across all Wan2.1 tasks)
    download(
        repo_id="Comfy-Org/Wan_2.1_ComfyUI_repackaged",
        filename="wan_2.1_vae.safetensors",
        subfolder="split_files/vae",
        local_dir=MODEL_DIR,
    )

    # T5 text encoder (shared across all Wan2.1 tasks)
    download(
        repo_id="Wan-AI/Wan2.1-T2V-1.3B",
        filename="models_t5_umt5-xxl-enc-bf16.pth",
        local_dir=MODEL_DIR,
    )

    # CLIP model (only needed for I2V)
    if "i2v" in ARCHITECTURE:
        download(
            repo_id="Wan-AI/Wan2.1-I2V-14B-720P",
            filename="models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth",
            local_dir=MODEL_DIR,
        )

    # DiT weights (architecture-dependent)
    dit_file_map = {
        "wan2.1_t2v_1.3B":      "wan2.1_t2v_1.3B_bf16.safetensors",
        "wan2.1_t2v_14B":       "wan2.1_t2v_14B_bf16.safetensors",
        "wan2.1_i2v_14B_480p":  "wan2.1_i2v_480p_14B_bf16.safetensors",
        "wan2.1_i2v_14B_720p":  "wan2.1_i2v_720p_14B_bf16.safetensors",
    }
    dit_filename = dit_file_map.get(ARCHITECTURE)
    if dit_filename:
        download(
            repo_id="Comfy-Org/Wan_2.1_ComfyUI_repackaged",
            filename=dit_filename,
            subfolder="split_files/diffusion_models",
            local_dir=MODEL_DIR,
        )
    else:
        print(f"  ⚠️  Unknown architecture: {ARCHITECTURE}. Add DiT filename manually.")

elif ARCHITECTURE == "hunyuan_video":
    print("=== Downloading HunyuanVideo models ===")

    # DiT — fp8 unofficial (smaller, works with --fp8_base)
    download(
        repo_id="kohya-ss/HunyuanVideo-fp8_e4m3fn-unofficial",
        filename="mp_rank_00_model_states_fp8.safetensors",
        local_dir=MODEL_DIR,
    )

    # VAE
    download(
        repo_id="tencent/HunyuanVideo",
        filename="pytorch_model.pt",
        subfolder="hunyuan-video-t2v-720p/vae",
        local_dir=MODEL_DIR,
    )

    # Text Encoder 1 — LLaVA-LLaMA3 (LLM, fp16)
    download(
        repo_id="Comfy-Org/HunyuanVideo_repackaged",
        filename="llava_llama3_fp16.safetensors",
        subfolder="split_files/text_encoders",
        local_dir=MODEL_DIR,
    )

    # Text Encoder 2 — CLIP-L
    download(
        repo_id="Comfy-Org/HunyuanVideo_repackaged",
        filename="clip_l.safetensors",
        subfolder="split_files/text_encoders",
        local_dir=MODEL_DIR,
    )

else:
    print(f"Unsupported architecture: {ARCHITECTURE}")

print("\n✅ Model download complete.")


## Section 7 — Dataset Configuration

### Upload your images and captions to Google Drive

Place files in `DATASET_DIR` (configured in Section 4). For each image, create a matching `.txt` caption file:

```
dataset/
├── image001.jpg
├── image001.txt     ← caption: "a photo of sks person smiling"
├── image002.png
├── image002.txt
└── ...
```

**Caption tips:**
- Use a unique trigger word (e.g., `sks`, `ohwx`, `TOK`) in each caption
- Keep captions descriptive: `"a photo of sks person walking in a park"`
- 10–30 images is usually enough for a style LoRA; 20–50 for a character LoRA

Run the cell below to generate the TOML dataset config file.


In [ ]:
import os, toml

# Install toml if not available
try:
    import toml
except ImportError:
    !pip install -q toml
    import toml

TOML_PATH = f"{DRIVE_ROOT}/dataset_config.toml"

config = {
    "general": {
        "resolution": RESOLUTION,
        "caption_extension": CAPTION_EXT,
        "batch_size": BATCH_SIZE,
        "enable_bucket": ENABLE_BUCKET,
        "bucket_no_upscale": False,
    },
    "datasets": []
}

if DATASET_TYPE == "image":
    ds_entry = {
        "image_directory": DATASET_DIR,
        "cache_directory": CACHE_DIR,
        "num_repeats": NUM_REPEATS,
    }
elif DATASET_TYPE == "video":
    ds_entry = {
        "video_directory": DATASET_DIR,
        "cache_directory": CACHE_DIR,
        "num_repeats": NUM_REPEATS,
        "target_frames": TARGET_FRAMES,
        "frame_extraction": FRAME_EXTRACTION,
        "max_frames": MAX_FRAMES,
    }
else:
    raise ValueError(f"Unknown DATASET_TYPE: {DATASET_TYPE}")

config["datasets"].append(ds_entry)

with open(TOML_PATH, "w") as f:
    toml.dump(config, f)

print(f"✅ Dataset config written to: {TOML_PATH}")
print()
print(open(TOML_PATH).read())

# Count dataset files
if os.path.exists(DATASET_DIR):
    images = [f for f in os.listdir(DATASET_DIR)
              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
    captions = [f for f in os.listdir(DATASET_DIR) if f.endswith(CAPTION_EXT)]
    print(f"Dataset: {len(images)} images, {len(captions)} caption files found.")
    if len(images) == 0:
        print("⚠️  No images found in DATASET_DIR. Upload your training data before caching.")
    if len(captions) < len(images):
        print(f"⚠️  {len(images) - len(captions)} images have no caption file.")
else:
    print(f"⚠️  DATASET_DIR does not exist: {DATASET_DIR}")


## Section 8 — Configure Accelerate

Writes the Accelerate config for single-GPU training (no user input required).


In [ ]:
import os, yaml

accelerate_config = {
    "compute_environment": "LOCAL_MACHINE",
    "debug": False,
    "distributed_type": "NO",
    "downcast_bf16": "no",
    "enable_cpu_affinity": False,
    "gpu_ids": "0",
    "machine_rank": 0,
    "main_training_function": "main",
    "mixed_precision": MIXED_PRECISION,
    "num_machines": 1,
    "num_processes": 1,
    "rdzv_backend": "static",
    "same_network": True,
    "tpu_env": [],
    "tpu_use_cluster": False,
    "tpu_use_sudo": False,
    "use_cpu": False,
}

accel_dir = os.path.expanduser("~/.cache/huggingface/accelerate")
os.makedirs(accel_dir, exist_ok=True)
accel_path = os.path.join(accel_dir, "default_config.yaml")

with open(accel_path, "w") as f:
    yaml.dump(accelerate_config, f)

print(f"✅ Accelerate config written to: {accel_path}")
print(f"   mixed_precision : {MIXED_PRECISION}")


## Section 9 — Cache Latents

Encodes your training images/videos through the VAE and saves the latents to disk.
This runs once and is reused for all training epochs.


In [ ]:
import os, subprocess

REPO_DIR = "/content/musubi-tuner"
os.chdir(REPO_DIR)

if ARCHITECTURE.startswith("wan2.1"):
    vae_path = os.path.join(MODEL_DIR, "wan_2.1_vae.safetensors")

    # I2V needs the CLIP model to embed the reference image into the latent space
    i2v_flag = "--i2v" if "i2v" in ARCHITECTURE else ""
    clip_flag = ""
    if "i2v" in ARCHITECTURE:
        clip_path = os.path.join(MODEL_DIR,
            "models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth")
        clip_flag = f"--clip {clip_path}"

    cmd = (
        f"python src/musubi_tuner/wan_cache_latents.py"
        f" --dataset_config {DRIVE_ROOT}/dataset_config.toml"
        f" --vae {vae_path}"
        f" {i2v_flag}"
        f" {clip_flag}"
        f" --vae_cache_cpu"          # saves VRAM on T4
    )

elif ARCHITECTURE == "hunyuan_video":
    vae_path = os.path.join(MODEL_DIR, "pytorch_model.pt")

    cmd = (
        f"python src/musubi_tuner/cache_latents.py"
        f" --dataset_config {DRIVE_ROOT}/dataset_config.toml"
        f" --vae {vae_path}"
        f" --vae_chunk_size 32"
        f" --vae_tiling"
        f" --vae_spatial_tile_sample_min_size 128"  # reduces VRAM usage
    )

else:
    raise ValueError(f"Unsupported architecture: {ARCHITECTURE}")

print("Running latent caching command:")
print(cmd)
print()
!{cmd}


## Section 10 — Cache Text Encoder Outputs

Encodes all captions through the text encoder(s) and caches the results.
Text encoders are large and only needed once for caching — they won't be loaded during training.


In [ ]:
import os

REPO_DIR = "/content/musubi-tuner"
os.chdir(REPO_DIR)

if ARCHITECTURE.startswith("wan2.1"):
    t5_path = os.path.join(MODEL_DIR, "models_t5_umt5-xxl-enc-bf16.pth")

    import torch
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    fp8_t5_flag = "--fp8_t5" if vram_gb < 20 else ""  # use fp8 T5 on T4

    cmd = (
        f"python src/musubi_tuner/wan_cache_text_encoder_outputs.py"
        f" --dataset_config {DRIVE_ROOT}/dataset_config.toml"
        f" --t5 {t5_path}"
        f" --batch_size 4"
        f" {fp8_t5_flag}"
    )

elif ARCHITECTURE == "hunyuan_video":
    llm_path  = os.path.join(MODEL_DIR, "llava_llama3_fp16.safetensors")
    clip_path = os.path.join(MODEL_DIR, "clip_l.safetensors")

    import torch
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    fp8_llm_flag = "--fp8_llm" if vram_gb < 20 else ""

    cmd = (
        f"python src/musubi_tuner/cache_text_encoder_outputs.py"
        f" --dataset_config {DRIVE_ROOT}/dataset_config.toml"
        f" --text_encoder1 {llm_path}"
        f" --text_encoder2 {clip_path}"
        f" --batch_size 4"
        f" {fp8_llm_flag}"
    )

else:
    raise ValueError(f"Unsupported architecture: {ARCHITECTURE}")

print("Running text encoder caching command:")
print(cmd)
print()
!{cmd}


## Section 11 — Train LoRA

Starts the actual training. Checkpoints are saved to `OUTPUT_DIR` every `SAVE_EVERY_N_EPOCHS` epochs.

### Monitoring
- Watch loss values printed to the cell output
- Enable TensorBoard by setting `USE_TENSORBOARD = True` in Section 4

### If you get OOM (Out of Memory) errors
1. Increase `BLOCKS_TO_SWAP` (e.g., 20 for 14B models, 36 for HunyuanVideo)
2. Set `GRADIENT_CHECKPOINTING_CPU_OFFLOAD = True`
3. Reduce `RESOLUTION` (e.g., `[512, 512]`)
4. Reduce `BATCH_SIZE` to 1
5. Reduce `NETWORK_DIM` to 16


In [ ]:
import os

REPO_DIR = "/content/musubi-tuner"
os.chdir(REPO_DIR)

# ---- Build architecture-specific flags ----
if ARCHITECTURE.startswith("wan2.1"):
    train_script = "src/musubi_tuner/wan_train_network.py"
    network_module = "networks.lora_wan"

    # Map architecture to --task value
    task_map = {
        "wan2.1_t2v_1.3B":     "t2v-1.3B",
        "wan2.1_t2v_14B":      "t2v-14B",
        "wan2.1_i2v_14B_480p": "i2v-14B",
        "wan2.1_i2v_14B_720p": "i2v-14B",
    }
    task = task_map.get(ARCHITECTURE, "t2v-1.3B")

    # DiT file names
    dit_file_map = {
        "wan2.1_t2v_1.3B":     "wan2.1_t2v_1.3B_bf16.safetensors",
        "wan2.1_t2v_14B":      "wan2.1_t2v_14B_bf16.safetensors",
        "wan2.1_i2v_14B_480p": "wan2.1_i2v_480p_14B_bf16.safetensors",
        "wan2.1_i2v_14B_720p": "wan2.1_i2v_720p_14B_bf16.safetensors",
    }
    dit_path = os.path.join(MODEL_DIR, dit_file_map[ARCHITECTURE])

    arch_flags = f"--task {task} --dit {dit_path}"

elif ARCHITECTURE == "hunyuan_video":
    train_script = "src/musubi_tuner/hv_train_network.py"
    network_module = "networks.lora"

    dit_path = os.path.join(MODEL_DIR, "mp_rank_00_model_states_fp8.safetensors")
    arch_flags = f"--dit {dit_path}"

else:
    raise ValueError(f"Unsupported architecture: {ARCHITECTURE}")

# ---- Memory optimization flags ----
mem_flags = []
if USE_FP8_BASE:
    mem_flags.append("--fp8_base")
if GRADIENT_CHECKPOINTING:
    mem_flags.append("--gradient_checkpointing")
if GRADIENT_CHECKPOINTING_CPU_OFFLOAD:
    mem_flags.append("--gradient_checkpointing_cpu_offload")
if BLOCKS_TO_SWAP > 0:
    mem_flags.append(f"--blocks_to_swap {BLOCKS_TO_SWAP}")

# ---- Logging flags ----
log_flags = ""
if USE_TENSORBOARD:
    log_dir = f"{OUTPUT_DIR}/logs"
    os.makedirs(log_dir, exist_ok=True)
    log_flags = f"--log_with tensorboard --logging_dir {log_dir}"

# ---- Full training command ----
cmd = (
    f"accelerate launch"
    f" --num_cpu_threads_per_process 1"
    f" --mixed_precision {MIXED_PRECISION}"
    f" {train_script}"
    f" {arch_flags}"
    f" --dataset_config {DRIVE_ROOT}/dataset_config.toml"
    f" --sdpa"
    f" --mixed_precision {MIXED_PRECISION}"
    f" {' '.join(mem_flags)}"
    f" --optimizer_type {OPTIMIZER_TYPE}"
    f" --learning_rate {LEARNING_RATE}"
    f" --max_data_loader_n_workers 2"
    f" --persistent_data_loader_workers"
    f" --network_module {network_module}"
    f" --network_dim {NETWORK_DIM}"
    f" --network_alpha {NETWORK_ALPHA}"
    f" --timestep_sampling {TIMESTEP_SAMPLING}"
    f" --discrete_flow_shift {DISCRETE_FLOW_SHIFT}"
    f" --max_train_epochs {MAX_TRAIN_EPOCHS}"
    f" --save_every_n_epochs {SAVE_EVERY_N_EPOCHS}"
    f" --seed {SEED}"
    f" --output_dir {OUTPUT_DIR}"
    f" --output_name {LORA_NAME}"
    f" {log_flags}"
)

print("Training command:")
print(cmd)
print()
print("Starting training... (checkpoints saved to OUTPUT_DIR every N epochs)")
print("="*60)
!{cmd}


## Section 12 — (Optional) Test Inference

Generate a test video/image using your trained LoRA to verify it's working.

Edit the `INFERENCE_*` variables below, then run the cell.


In [ ]:
import os, glob

REPO_DIR = "/content/musubi-tuner"
os.chdir(REPO_DIR)

# ---- Inference settings ----
INFER_PROMPT      = "a photo of sks person walking in a sunny park, cinematic"  # EDIT THIS
INFER_STEPS       = 20
INFER_SEED        = 42
INFER_VIDEO_SIZE  = [832, 480]   # [W, H]
INFER_VIDEO_LEN   = 1            # 1 for single frame (image), N*4+1 for video
INFER_OUTPUT_DIR  = f"{DRIVE_ROOT}/inference_output"

os.makedirs(INFER_OUTPUT_DIR, exist_ok=True)

# Auto-select the latest LoRA checkpoint
lora_files = sorted(
    glob.glob(f"{OUTPUT_DIR}/{LORA_NAME}*.safetensors"),
    key=os.path.getmtime
)
if not lora_files:
    print(f"⚠️  No LoRA files found in {OUTPUT_DIR}. Train first or set LORA_PATH manually.")
    LORA_PATH = None
else:
    LORA_PATH = lora_files[-1]  # use most recent
    print(f"Using LoRA: {LORA_PATH}")

if LORA_PATH:
    if ARCHITECTURE.startswith("wan2.1"):
        t5_path  = os.path.join(MODEL_DIR, "models_t5_umt5-xxl-enc-bf16.pth")
        vae_path = os.path.join(MODEL_DIR, "wan_2.1_vae.safetensors")
        dit_file_map = {
            "wan2.1_t2v_1.3B":     "wan2.1_t2v_1.3B_bf16.safetensors",
            "wan2.1_t2v_14B":      "wan2.1_t2v_14B_bf16.safetensors",
            "wan2.1_i2v_14B_480p": "wan2.1_i2v_480p_14B_bf16.safetensors",
            "wan2.1_i2v_14B_720p": "wan2.1_i2v_720p_14B_bf16.safetensors",
        }
        task_map = {
            "wan2.1_t2v_1.3B":     "t2v-1.3B",
            "wan2.1_t2v_14B":      "t2v-14B",
            "wan2.1_i2v_14B_480p": "i2v-14B",
            "wan2.1_i2v_14B_720p": "i2v-14B",
        }
        dit_path = os.path.join(MODEL_DIR, dit_file_map[ARCHITECTURE])
        task     = task_map[ARCHITECTURE]

        cmd = (
            f"python src/musubi_tuner/wan_generate_video.py"
            f" --fp8"
            f" --task {task}"
            f" --dit {dit_path}"
            f" --vae {vae_path}"
            f" --t5 {t5_path}"
            f" --attn_mode torch"
            f" --video_size {INFER_VIDEO_SIZE[0]} {INFER_VIDEO_SIZE[1]}"
            f" --video_length {INFER_VIDEO_LEN}"
            f" --infer_steps {INFER_STEPS}"
            f" --prompt \"{INFER_PROMPT}\""
            f" --save_path {INFER_OUTPUT_DIR}/test_output.mp4"
            f" --output_type both"
            f" --seed {INFER_SEED}"
            f" --lora_multiplier 1.0"
            f" --lora_weight {LORA_PATH}"
        )

    elif ARCHITECTURE == "hunyuan_video":
        dit_path  = os.path.join(MODEL_DIR, "mp_rank_00_model_states_fp8.safetensors")
        vae_path  = os.path.join(MODEL_DIR, "pytorch_model.pt")
        llm_path  = os.path.join(MODEL_DIR, "llava_llama3_fp16.safetensors")
        clip_path = os.path.join(MODEL_DIR, "clip_l.safetensors")

        cmd = (
            f"python src/musubi_tuner/hv_generate_video.py"
            f" --fp8"
            f" --dit {dit_path}"
            f" --vae {vae_path}"
            f" --text_encoder1 {llm_path}"
            f" --text_encoder2 {clip_path}"
            f" --attn_mode sdpa --split_attn"
            f" --video_size {INFER_VIDEO_SIZE[1]} {INFER_VIDEO_SIZE[0]}"
            f" --video_length {INFER_VIDEO_LEN}"
            f" --infer_steps {INFER_STEPS}"
            f" --prompt \"{INFER_PROMPT}\""
            f" --save_path {INFER_OUTPUT_DIR}/test_output.mp4"
            f" --output_type both"
            f" --vae_chunk_size 32 --vae_spatial_tile_sample_min_size 128"
            f" --seed {INFER_SEED}"
            f" --lora_multiplier 1.0"
            f" --lora_weight {LORA_PATH}"
        )

    print("Inference command:")
    print(cmd)
    print()
    !{cmd}

    # Try to display the output if it's an image
    import glob
    out_images = glob.glob(f"{INFER_OUTPUT_DIR}/*.png") + glob.glob(f"{INFER_OUTPUT_DIR}/*.jpg")
    if out_images:
        from IPython.display import Image, display
        display(Image(out_images[-1]))


## Section 13 — List & Download LoRA Files

Your LoRA files are already in Google Drive (`OUTPUT_DIR`). Use this cell to list them and optionally download to your local machine.


In [ ]:
import os, glob
from IPython.display import FileLink, display

lora_files = sorted(
    glob.glob(f"{OUTPUT_DIR}/*.safetensors"),
    key=os.path.getmtime
)

if not lora_files:
    print("No LoRA files found in OUTPUT_DIR. Train first.")
else:
    print(f"Found {len(lora_files)} LoRA checkpoint(s) in {OUTPUT_DIR}:")
    for f in lora_files:
        size_mb = os.path.getsize(f) / 1024**2
        print(f"  {os.path.basename(f)}  ({size_mb:.1f} MB)")

    print()
    print("Files are in your Google Drive and can be accessed from there.")
    print("To download the latest checkpoint to your browser, run the cell below:")

# Optionally download the latest checkpoint directly to your browser
# (Uncomment this block)
# from google.colab import files
# if lora_files:
#     files.download(lora_files[-1])


## Tips & Troubleshooting

### Out of Memory (OOM)
- Increase `BLOCKS_TO_SWAP` in Section 4 (try 20 for Wan 14B, 36 for HunyuanVideo)
- Set `GRADIENT_CHECKPOINTING_CPU_OFFLOAD = True`
- Reduce resolution to `[512, 512]` or `[640, 360]`
- Reduce `NETWORK_DIM` to 16 or 8
- Switch to Wan2.1 T2V 1.3B if using a 14B or HunyuanVideo model on T4

### Poor training results
- Increase dataset size (aim for 15–50 images for character LoRA)
- Add a unique trigger word to all captions (e.g., `ohwx person`)
- Try `LEARNING_RATE = 1e-4` or `5e-4`
- Increase `MAX_TRAIN_EPOCHS` to 20–32
- Try `NETWORK_DIM = 64` for more capacity

### Colab disconnects mid-training
- All checkpoints are saved to Google Drive — just re-run from Section 11
- Re-run Sections 3, 4, 5 to restore the environment (models are already on Drive)

### Convert LoRA for ComfyUI
```bash
python src/musubi_tuner/convert_lora.py \
  --input /path/to/musubi_lora.safetensors \
  --output /path/to/comfyui_lora.safetensors \
  --target other
```

### Resources
- [musubi-tuner GitHub](https://github.com/kohya-ss/musubi-tuner)
- [Wan2.1 training docs](https://github.com/kohya-ss/musubi-tuner/blob/main/docs/wan.md)
- [HunyuanVideo training docs](https://github.com/kohya-ss/musubi-tuner/blob/main/docs/hunyuan_video.md)
- [Dataset config docs](https://github.com/kohya-ss/musubi-tuner/blob/main/docs/dataset_config.md)
